In [1]:
!pip install scipy
# ============================================================
# EPU ATTENTION — QUESTION 2 ONLY (PURE LENGTH VERSION)
# ------------------------------------------------------------
# Question 2:
# Does performance degrade systematically as input length increases,
# even when task difficulty is held constant?
#
# Design:
# - Same sampled articles for all models and all conditions
# - SAME target article text in all conditions
# - SAME label in all conditions
# - SAME prompt instructions in all conditions
# - Only total input length changes
# - Length is increased with neutral archive-style filler,
#   not salient distractors and not other articles
# - Score = ONLY main-label EPU accuracy
# ============================================================

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import kaggle_benchmarks as kbench

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600

# Keep the SAME target article text across short/medium/long
TARGET_MAX_CHARS = 3000

# Character budgets for total prompt-side article dossier length
SHORT_TOTAL_CHARS = TARGET_MAX_CHARS          # target only
MEDIUM_TOTAL_CHARS = 7000                     # target + neutral filler
LONG_TOTAL_CHARS = 13000                      # target + more neutral filler

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    # "qwen/qwen3-235b-a22b-instruct-2507",
]

OUT_DIR = Path("/kaggle/working/epu_attention_q2_pure_length_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]

for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)

clean = df.loc[mask].copy()

clean["article_key"] = clean["article_key"].astype(str)
missing_key_mask = clean["article_key"].isin(["nan", "None", ""])
if missing_key_mask.any():
    clean.loc[missing_key_mask, "article_key"] = [
        f"generated_key_{i}" for i in range(missing_key_mask.sum())
    ]

clean["gold_epu"] = clean["gold_epu"].astype(int)
clean["disagreement_flag"] = clean["disagreement_flag"].fillna(0).astype(int)
clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
print("\nClass balance:")
print(clean["gold_epu"].value_counts().sort_index())
print("\nDisagreement rows available:")
print(clean["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# BALANCED SAMPLE
# ---------------------------
def balanced_sample(data: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()

    if len(pos) == 0 or len(neg) == 0:
        return data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)

    n_pos = min(len(pos), n // 2)
    n_neg = min(len(neg), n - n_pos)

    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed + 1)

    pilot = (
        pd.concat([pos_s, neg_s], axis=0)
        .sample(frac=1, random_state=seed + 2)
        .reset_index(drop=True)
    )
    return pilot

pilot = balanced_sample(clean, N_ROWS, SEED).copy()

print("\nPilot preview:")
preview_cols = [c for c in ["article_key", "gold_epu", "disagreement_flag", "newspaper", "year", "content_len"] if c in pilot.columns]
display(pilot[preview_cols].head())

print("\nPilot class balance:")
print(pilot["gold_epu"].value_counts().sort_index())

print("\nPilot disagreement balance:")
print(pilot["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# PURE LENGTH FILLER
# ---------------------------
# Neutral archive-style filler. No economics/politics/policy/task guidance.
NEUTRAL_FILLER_BLOCK = (
    "Archive page marker. Section header. Edition note. Library index line. "
    "Continuation marker. Photo caption omitted. Layout reference. Page sequence note. "
    "Archive copy marker. Section listing. Internal reference line. Continuation note. "
)

def smart_truncate(text: str, max_chars: int = TARGET_MAX_CHARS) -> str:
    text = str(text)
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head - 32
    return text[:head] + "\n\n[...TRUNCATED...]\n\n" + text[-tail:]

def make_filler(n_chars: int) -> str:
    if n_chars <= 0:
        return ""
    reps = (n_chars // len(NEUTRAL_FILLER_BLOCK)) + 2
    filler = (NEUTRAL_FILLER_BLOCK * reps)[:n_chars]
    return filler.strip()

def split_filler(total_filler_chars: int) -> tuple[str, str]:
    pre_chars = total_filler_chars // 2
    post_chars = total_filler_chars - pre_chars
    return make_filler(pre_chars), make_filler(post_chars)

pilot["content_for_eval"] = pilot["content"].astype(str).apply(lambda x: smart_truncate(x, TARGET_MAX_CHARS))
pilot["target_chars"] = pilot["content_for_eval"].astype(str).str.len()

def add_length_blocks(row):
    target_chars = int(row["target_chars"])

    short_filler_total = max(0, SHORT_TOTAL_CHARS - target_chars)
    medium_filler_total = max(0, MEDIUM_TOTAL_CHARS - target_chars)
    long_filler_total = max(0, LONG_TOTAL_CHARS - target_chars)

    short_pre, short_post = split_filler(short_filler_total)
    medium_pre, medium_post = split_filler(medium_filler_total)
    long_pre, long_post = split_filler(long_filler_total)

    return pd.Series({
        "short_pre": short_pre,
        "short_post": short_post,
        "medium_pre": medium_pre,
        "medium_post": medium_post,
        "long_pre": long_pre,
        "long_post": long_post,
        "short_total_chars": target_chars + short_filler_total,
        "medium_total_chars": target_chars + medium_filler_total,
        "long_total_chars": target_chars + long_filler_total,
    })

pilot = pd.concat([pilot, pilot.apply(add_length_blocks, axis=1)], axis=1)

pilot_path = OUT_DIR / "pilot_rows_q2.csv"
pilot.to_csv(pilot_path, index=False)

# ---------------------------
# HELPERS
# ---------------------------
def maybe_int(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return int(x)
    except Exception:
        return None

def binary_match(pred, gold):
    gold_i = maybe_int(gold)
    pred_i = maybe_int(pred)
    if gold_i is None:
        return None
    if pred_i is None:
        return 0.0
    return 1.0 if pred_i == gold_i else 0.0

def default_out():
    return {
        "epu": None,
        "confidence": 0.0,
        "focus_excerpt": "",
    }

def parse_llm_json(raw_text):
    out = default_out()

    if raw_text is None:
        return out

    text = str(raw_text).strip()
    text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            return out
    except Exception:
        return out

    for k in out:
        if k in obj:
            out[k] = obj[k]

    out["epu"] = maybe_int(out["epu"])

    try:
        out["confidence"] = float(out["confidence"])
    except Exception:
        out["confidence"] = 0.0

    out["confidence"] = max(0.0, min(1.0, out["confidence"]))
    out["focus_excerpt"] = "" if out["focus_excerpt"] is None else str(out["focus_excerpt"])[:250]

    return out

def model_json_call(llm, prompt: str):
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception:
        return default_out()

def build_q2_prompt(article_text: str, context_pre: str = "", context_post: str = "") -> str:
    pre_block = f"{context_pre}\n\n" if context_pre else ""
    post_block = f"\n\n{context_post}" if context_post else ""

    return (
        "You are evaluating only the text between TARGET ARTICLE START and TARGET ARTICLE END.\n\n"
        "Task:\n"
        "Decide whether the TARGET ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).\n\n"
        "Labeling rule:\n"
        "- EPU = 1 only if the TARGET ARTICLE discusses uncertainty about government policy,\n"
        "  policy decisions, possible policy actions, political outcomes affecting policy,\n"
        "  or the economic effects of policy.\n"
        "- EPU = 0 for general economics, elections, markets, war, uncertainty, or foreign events\n"
        "  unless the uncertainty is specifically tied to economic policy.\n\n"
        "IMPORTANT:\n"
        "- Use only the text inside TARGET ARTICLE START and TARGET ARTICLE END for the decision.\n"
        "- Return ONLY valid JSON.\n"
        "- No markdown. No explanation. No code fence.\n\n"
        "Return exactly these keys:\n"
        "{\n"
        '  "epu": 0 or 1,\n'
        '  "confidence": number between 0 and 1,\n'
        '  "focus_excerpt": "short quote from target article only"\n'
        "}\n\n"
        f"{pre_block}"
        "TARGET ARTICLE START\n"
        f"{article_text}\n"
        "TARGET ARTICLE END"
        f"{post_block}"
    ).strip()

# ---------------------------
# Q2 TASKS
# ---------------------------
@kbench.task(name="epu_q2_short")
def epu_q2_short(
    llm,
    article_key: str,
    content_for_eval: str,
    gold_epu: int,
    short_pre: str = "",
    short_post: str = "",
    medium_pre: str = "",
    medium_post: str = "",
    long_pre: str = "",
    long_post: str = "",
    disagreement_flag: int = 0,
) -> float:
    out = model_json_call(llm, build_q2_prompt(content_for_eval, short_pre, short_post))
    m = binary_match(out["epu"], gold_epu)
    return 0.0 if m is None else float(m)

@kbench.task(name="epu_q2_medium")
def epu_q2_medium(
    llm,
    article_key: str,
    content_for_eval: str,
    gold_epu: int,
    short_pre: str = "",
    short_post: str = "",
    medium_pre: str = "",
    medium_post: str = "",
    long_pre: str = "",
    long_post: str = "",
    disagreement_flag: int = 0,
) -> float:
    out = model_json_call(llm, build_q2_prompt(content_for_eval, medium_pre, medium_post))
    m = binary_match(out["epu"], gold_epu)
    return 0.0 if m is None else float(m)

@kbench.task(name="epu_q2_long")
def epu_q2_long(
    llm,
    article_key: str,
    content_for_eval: str,
    gold_epu: int,
    short_pre: str = "",
    short_post: str = "",
    medium_pre: str = "",
    medium_post: str = "",
    long_pre: str = "",
    long_post: str = "",
    disagreement_flag: int = 0,
) -> float:
    out = model_json_call(llm, build_q2_prompt(content_for_eval, long_pre, long_post))
    m = binary_match(out["epu"], gold_epu)
    return 0.0 if m is None else float(m)

# ---------------------------
# EVAL DATA
# ---------------------------
eval_cols = [
    "article_key",
    "content_for_eval",
    "gold_epu",
    "short_pre",
    "short_post",
    "medium_pre",
    "medium_post",
    "long_pre",
    "long_post",
    "disagreement_flag",
]

eval_df = pilot[eval_cols].copy()
eval_df["gold_epu"] = eval_df["gold_epu"].astype(int)
eval_df["disagreement_flag"] = eval_df["disagreement_flag"].fillna(0).astype(int)

assert eval_df.isna().sum().sum() == 0, "eval_df still contains NaN values"

print("\nEvaluation dataframe shape:", eval_df.shape)
print("Total NaN in eval_df:", int(eval_df.isna().sum().sum()))
print("\nAverage total chars by condition:")
print(pd.Series({
    "short": pilot["short_total_chars"].mean(),
    "medium": pilot["medium_total_chars"].mean(),
    "long": pilot["long_total_chars"].mean(),
}).round(1))

# ---------------------------
# RUN MODELS × 3 CONDITIONS
# ---------------------------
llm_grid = [kbench.llms[name] for name in MODEL_NAMES]

print("\nRunning models:")
for name in MODEL_NAMES:
    print(" -", name)

def normalize_results(results_obj, condition_name):
    df_out = results_obj.as_dataframe().copy().reset_index()
    if "llm" in df_out.columns:
        df_out["llm_name"] = df_out["llm"].astype(str)
        df_out = df_out.drop(columns=["llm"])
    else:
        df_out["llm_name"] = "unknown_model"
    df_out["condition"] = condition_name
    return df_out

res_short = epu_q2_short.evaluate(llm=llm_grid, evaluation_data=eval_df)
res_medium = epu_q2_medium.evaluate(llm=llm_grid, evaluation_data=eval_df)
res_long = epu_q2_long.evaluate(llm=llm_grid, evaluation_data=eval_df)

results_df = pd.concat(
    [
        normalize_results(res_short, "short"),
        normalize_results(res_medium, "medium"),
        normalize_results(res_long, "long"),
    ],
    ignore_index=True,
)

print("\nresults_df columns:")
print(results_df.columns.tolist())

display_cols = [c for c in ["run_id", "llm_name", "condition", "article_key", "gold_epu", "disagreement_flag", "result", "id"] if c in results_df.columns]
display(results_df[display_cols].head(12))

# ---------------------------
# SAVE RAW RESULTS
# ---------------------------
raw_csv = OUT_DIR / "q2_results_raw.csv"
results_df.to_csv(raw_csv, index=False)

# ---------------------------
# MAIN Q2 OUTPUTS
# ---------------------------
summary_q2 = (
    results_df.groupby(["llm_name", "condition"], sort=False)["result"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary_q2_csv = OUT_DIR / "q2_summary_overall.csv"
summary_q2.to_csv(summary_q2_csv, index=False)

degradation_table = (
    summary_q2.pivot(index="llm_name", columns="condition", values="mean")
    .reset_index()
)
for col in ["short", "medium", "long"]:
    if col not in degradation_table.columns:
        degradation_table[col] = np.nan

degradation_table["medium_minus_short"] = degradation_table["medium"] - degradation_table["short"]
degradation_table["long_minus_short"] = degradation_table["long"] - degradation_table["short"]
degradation_table["long_minus_medium"] = degradation_table["long"] - degradation_table["medium"]
degradation_table_csv = OUT_DIR / "q2_degradation_table.csv"
degradation_table.to_csv(degradation_table_csv, index=False)

summary_by_label = (
    results_df.groupby(["llm_name", "condition", "gold_epu"], sort=False)["result"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary_by_label_csv = OUT_DIR / "q2_summary_by_gold_epu.csv"
summary_by_label.to_csv(summary_by_label_csv, index=False)

summary_by_disagreement = (
    results_df.groupby(["llm_name", "condition", "disagreement_flag"], sort=False)["result"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary_by_disagreement_csv = OUT_DIR / "q2_summary_by_disagreement.csv"
summary_by_disagreement.to_csv(summary_by_disagreement_csv, index=False)

hardest_rows = (
    results_df.groupby(["article_key", "gold_epu", "disagreement_flag", "condition"], sort=False)["result"]
    .agg(["mean", "std", "count", "min", "max"])
    .reset_index()
    .sort_values(["condition", "mean", "std"], ascending=[True, True, False])
)
snippet_df = pilot[["article_key", "content_for_eval"]].copy()
snippet_df["snippet"] = snippet_df["content_for_eval"].astype(str).str.slice(0, 220)
snippet_df = snippet_df[["article_key", "snippet"]]
hardest_rows = hardest_rows.merge(snippet_df, on="article_key", how="left")
hardest_rows_csv = OUT_DIR / "q2_hardest_rows.csv"
hardest_rows.to_csv(hardest_rows_csv, index=False)

article_model_matrix = results_df.pivot_table(
    index=["article_key", "gold_epu", "disagreement_flag", "condition"],
    columns="llm_name",
    values="result",
    aggfunc="mean",
).reset_index()
article_model_matrix_csv = OUT_DIR / "q2_article_model_matrix.csv"
article_model_matrix.to_csv(article_model_matrix_csv, index=False)

manifest = pd.DataFrame({
    "file": [
        raw_csv.name,
        summary_q2_csv.name,
        degradation_table_csv.name,
        summary_by_label_csv.name,
        summary_by_disagreement_csv.name,
        hardest_rows_csv.name,
        article_model_matrix_csv.name,
        pilot_path.name,
    ],
    "path": [
        str(raw_csv),
        str(summary_q2_csv),
        str(degradation_table_csv),
        str(summary_by_label_csv),
        str(summary_by_disagreement_csv),
        str(hardest_rows_csv),
        str(article_model_matrix_csv),
        str(pilot_path),
    ],
})
manifest_csv = OUT_DIR / "manifest.csv"
manifest.to_csv(manifest_csv, index=False)

print("\nSaved files:")
for p in [raw_csv, summary_q2_csv, degradation_table_csv, summary_by_label_csv,
          summary_by_disagreement_csv, hardest_rows_csv, article_model_matrix_csv,
          pilot_path, manifest_csv]:
    print(p)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/35.3 MB ? eta -:--:--

   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/35.3 MB 12.0 MB/s eta 0:00:03

   ━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/35.3 MB 20.1 MB/s eta 0:00:02

   ━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━ 20.4/35.3 MB 35.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 35.1/35.3 MB 49.8 MB/s eta 0:00:01

...

[Parallel(n_jobs=1)]: Done 511 tasks      | elapsed: 22.4min



results_df columns:
['run_id', 'article_key', 'content_for_eval', 'gold_epu', 'short_pre', 'short_post', 'medium_pre', 'medium_post', 'long_pre', 'long_post', 'disagreement_flag', 'result', 'id', 'llm_name', 'condition']


[Parallel(n_jobs=1)]: Done 600 out of 600 | elapsed: 26.3min finished


,run_id,llm_name,condition,article_key,gold_epu,disagreement_flag,result,id
0,Run #1,🤖 anthropic/claude-sonnet-4-6@default,short,Current_1332,1,0,0.0,0
1,Run #2,🤖 anthropic/claude-sonnet-4-6@default,short,Current_910,0,0,1.0,1
2,Run #3,🤖 anthropic/claude-sonnet-4-6@default,short,Midcentury_7857,0,0,0.0,2
3,Run #4,🤖 anthropic/claude-sonnet-4-6@default,short,Current_411,1,0,1.0,3
4,Run #5,🤖 anthropic/claude-sonnet-4-6@default,short,Current_769,1,0,0.0,4
5,Run #6,🤖 anthropic/claude-sonnet-4-6@default,short,Current_98,1,1,0.0,5
6,Run #7,🤖 anthropic/claude-sonnet-4-6@default,short,Current_58,1,1,1.0,6
7,Run #8,🤖 anthropic/claude-sonnet-4-6@default,short,Current_1440,0,0,1.0,7
8,Run #9,🤖 anthropic/claude-sonnet-4-6@default,short,Current_1870,1,0,1.0,8
9,Run #10,🤖 anthropic/claude-sonnet-4-6@default,short,Midcentury_5357,1,0,1.0,9



Saved files:
/kaggle/working/epu_attention_q2_pure_length_outputs/q2_results_raw.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/q2_summary_overall.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/q2_degradation_table.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/q2_summary_by_gold_epu.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/q2_summary_by_disagreement.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/q2_hardest_rows.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/q2_article_model_matrix.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/pilot_rows_q2.csv
/kaggle/working/epu_attention_q2_pure_length_outputs/manifest.csv
